# Notebook 11 — Validation Analysis (Remote Sensing Cross-Validation)
**Stage 11 of the Ghana Groundwater–Subsidence Pipeline**

Validates pipeline outputs using remote-sensing cross-validation. When borehole or institutional data become available, add them to `data/validation/` and re-run.

**Sections:**
1. Temporal validation — GWSA vs PDSI and CHIRPS (lag correlations)
2. Drought episode cross-reference — SGWA/PDSI episodes vs published Ghana drought chronology
3. Urban growth consistency — GHSL built-up vs population density
4. Indicator independence — Spearman correlation matrix of CGSRI inputs
5. CGSRI sensitivity — rank stability under 5 weighting schemes
6. Spatial pattern validation — CGSRI vs elevation, coast distance, population
7. Validation report and uncertainty summary

**Outputs:** 4 figures, 5 tables, `docs/validation_report.md`

In [ ]:
import os
import pathlib; os.chdir(str(pathlib.Path.cwd().parent) if pathlib.Path.cwd().name == 'notebooks' else str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from scipy.stats import spearmanr, pearsonr
from scipy import stats

ROOT = Path('.')
GW   = ROOT / 'data' / 'processed' / 'groundwater_anomalies'
RI   = ROOT / 'data' / 'processed' / 'risk_index'
CLI  = ROOT / 'data' / 'processed' / 'climate_indices'
TS   = ROOT / 'data' / 'interim' / 'time_series'
FIGS = ROOT / 'outputs' / 'figures'
TABS = ROOT / 'outputs' / 'tables'
DOCS = ROOT / 'docs'
FIGS.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'font.family':'sans-serif','font.size':9,
                     'savefig.dpi':150,'savefig.bbox':'tight'})

# ── Load core data ─────────────────────────────────────────────────────────
gdf = gpd.read_file(RI / 'cgsri_grid10km.gpkg')

df_raw = pd.read_csv(TS / 'gwsa_monthly.csv')
tc = 'date' if 'date' in df_raw.columns else 'time'
df_raw[tc] = pd.to_datetime(df_raw[tc])
gwsa_ts = df_raw.set_index(tc)['gwsa_mm']

tc_file   = CLI / 'terraclimate_coastal_timeseries.csv'
rain_file = CLI / 'rainfall_anomaly_monthly.csv'
df_tc   = pd.read_csv(tc_file,   parse_dates=['date']).set_index('date') if tc_file.exists() else None
df_rain = pd.read_csv(rain_file, parse_dates=['date']).set_index('date') if rain_file.exists() else None

print(f'CGSRI cells: {len(gdf)}')
print(f'GWSA months: {len(gwsa_ts)}')
print(f'TerraClimate: {df_tc is not None}  CHIRPS: {df_rain is not None}')

## 1. Temporal Validation — GWSA vs Climate Drivers

In [ ]:
# ── Align all time series ──────────────────────────────────────────────────
df_ts = pd.DataFrame({'gwsa_mm': gwsa_ts})
if df_tc is not None:
    for col in ['pdsi','precip_mm','pet_mm']:
        if col in df_tc.columns:
            df_ts[col] = df_tc[col].reindex(df_ts.index, method='nearest', tolerance='35D')
    if 'precip_mm' in df_ts and 'pet_mm' in df_ts:
        df_ts['p_minus_pet'] = df_ts['precip_mm'] - df_ts['pet_mm']
elif df_rain is not None:
    df_ts['precip_mm'] = df_rain['precip_mm'].reindex(df_ts.index, method='nearest', tolerance='35D')

# ── Lag correlation (0-12 months) ─────────────────────────────────────────
lag_results = []
for var in ['pdsi','precip_mm','p_minus_pet']:
    if var not in df_ts.columns: continue
    for lag in range(0, 13):
        shifted = df_ts[var].shift(lag)
        mask = ~(df_ts['gwsa_mm'].isna() | shifted.isna())
        if mask.sum() < 20: continue
        r, p = pearsonr(df_ts['gwsa_mm'][mask], shifted[mask])
        lag_results.append({'variable':var, 'lag_months':lag, 'r':round(r,4), 'p':round(p,5)})

df_lag = pd.DataFrame(lag_results)
print('Best lag (max |r|) per variable:')
for var in df_lag['variable'].unique():
    sub = df_lag[df_lag['variable']==var]
    best = sub.loc[sub['r'].abs().idxmax()]
    print(f'  {var:15s}  lag={int(best.lag_months):+d} mo  r={best.r:.3f}  p={best.p:.4f}')

print('\nConcurrent (lag=0) correlations:')
corr_summary = {}
for var in ['pdsi','precip_mm','p_minus_pet']:
    if var not in df_ts.columns: continue
    mask = ~(df_ts['gwsa_mm'].isna() | df_ts[var].isna())
    rp, pp = pearsonr(df_ts['gwsa_mm'][mask], df_ts[var][mask])
    rs, ps = spearmanr(df_ts['gwsa_mm'][mask], df_ts[var][mask])
    corr_summary[var] = (rp, pp, rs, ps)
    print(f'  {var:15s}  Pearson r={rp:.3f} (p={pp:.4f})  Spearman ρ={rs:.3f} (p={ps:.4f})')

df_lag.to_csv(TABS / 'validation_lag_correlations.csv', index=False)
print('Saved validation_lag_correlations.csv')

In [ ]:
# ── Temporal validation figure ─────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
fig.suptitle('Temporal Validation: GWSA vs Climate Drivers — Coastal Ghana',
             fontweight='bold', fontsize=11)

ax = axes[0]
ax.fill_between(df_ts.index, np.where(df_ts['gwsa_mm']<0, df_ts['gwsa_mm'], 0), 0,
                color='#CC4444', alpha=0.35, label='Deficit')
ax.fill_between(df_ts.index, np.where(df_ts['gwsa_mm']>=0, df_ts['gwsa_mm'], 0), 0,
                color='#4477AA', alpha=0.35, label='Surplus')
ax.plot(df_ts.index, df_ts['gwsa_mm'].rolling(6,center=True).mean(),
        color='navy', lw=1.4, label='6-month mean')
ax.axhline(0, color='k', lw=0.7, ls='--')
ax.set_ylabel('GWSA (mm)'); ax.legend(fontsize=7, ncol=3)
ax.set_title('(a) Groundwater storage anomaly (GRACE–GLDAS)', fontsize=9)

if 'pdsi' in df_ts.columns:
    ax = axes[1]
    ax.fill_between(df_ts.index, np.where(df_ts['pdsi']<0, df_ts['pdsi'], 0), 0,
                    color='#FF6B35', alpha=0.4)
    ax.fill_between(df_ts.index, np.where(df_ts['pdsi']>=0, df_ts['pdsi'], 0), 0,
                    color='#4682B4', alpha=0.4)
    ax.plot(df_ts.index, df_ts['pdsi'].rolling(3,center=True).mean(),
            color='darkred', lw=1.2)
    ax.axhline(-1, color='orange', lw=0.7, ls=':')
    ax.axhline(0, color='k', lw=0.7, ls='--')
    ax.set_ylim(-6, 6)
    rp, pp, rs, ps = corr_summary.get('pdsi',(np.nan,)*4)
    ax.set_title(f'(b) PDSI — Pearson r={rp:.3f} with GWSA (p={pp:.4f})', fontsize=9)
    ax.set_ylabel('PDSI')
else:
    axes[1].text(0.5,0.5,'PDSI not available',ha='center',va='center',
                 transform=axes[1].transAxes,color='gray')
    axes[1].set_title('(b) PDSI (not available)', fontsize=9)

if 'precip_mm' in df_ts.columns:
    ax = axes[2]
    ax.bar(df_ts.index, df_ts['precip_mm'], width=25, color='steelblue', alpha=0.45, label='Monthly')
    ax.plot(df_ts.index, df_ts['precip_mm'].rolling(12,center=True).mean(),
            color='navy', lw=1.4, label='12-month mean')
    rp, pp, rs, ps = corr_summary.get('precip_mm',(np.nan,)*4)
    ax.set_title(f'(c) CHIRPS — Pearson r={rp:.3f} with GWSA (p={pp:.4f})', fontsize=9)
    ax.set_ylabel('Precipitation (mm month⁻¹)')
    ax.legend(fontsize=7)
else:
    axes[2].text(0.5,0.5,'Rainfall not available',ha='center',va='center',
                 transform=axes[2].transAxes,color='gray')
axes[2].set_xlabel('Year')

plt.tight_layout()
fig.savefig(FIGS / 'validation_gwsa_vs_climate.png', dpi=150)
plt.close()
print('Saved validation_gwsa_vs_climate.png')

## 2. Drought Episode Cross-Reference

In [ ]:
# Published Ghana drought years (FEWS NET, OCHA, literature)
known = {
    1983: 'Major West Africa drought — crop failure, emergency aid',
    1994: 'Severe drought — food insecurity, WFP emergency response',
    2002: 'Northern Ghana drought — emergency declared',
    2007: 'Post-flood drought stress — northern Ghana',
    2011: 'Drought and flooding — northern Ghana, FEWS NET alert',
    2016: 'Below-normal rains — FEWS NET stress warning',
    2021: 'Dry spell — northern and central Ghana',
}

records = []
for ep_file, source in [(TABS/'drought_episodes.csv','SGWA'),
                         (TABS/'drought_episodes_pdsi.csv','PDSI')]:
    if not ep_file.exists(): continue
    df_ep = pd.read_csv(ep_file)
    print(f'\n{source} episodes ({len(df_ep)}):')
    print(df_ep.to_string(index=False))
    for _, row in df_ep.iterrows():
        start_str = str(row.iloc[0])
        yr = int(start_str[:4])
        lit = known.get(yr, known.get(yr+1, known.get(yr-1, '— No direct literature match')))
        rec = {'source': source, 'start': start_str}
        if 'end' in row: rec['end'] = row['end']
        if 'duration_months' in row: rec['duration_months'] = row['duration_months']
        if 'min_pdsi' in row: rec['min_index'] = row['min_pdsi']
        if 'severity' in row: rec['severity'] = row['severity']
        rec['literature_match'] = lit
        records.append(rec)

df_xref = pd.DataFrame(records)
df_xref.to_csv(TABS / 'validation_drought_crossref.csv', index=False)
print('\nSaved validation_drought_crossref.csv')

## 3. Urban Growth Consistency Check

In [ ]:
pairs = [
    ('builtup_density_2020', 'pop_density_km2',  'Built-up density vs pop density'),
    ('n_BD',                 'n_PD',             'Norm. building density vs pop density (CGSRI)'),
    ('builtup_growth_pct',   'pop_growth_rate',  'Built-up growth % vs pop growth rate'),
    ('n_BUGR',               'n_BD',             'Urban growth rate vs current density (sprawl)'),
    ('dist_coast_km',        'builtup_density_2020', 'Distance to coast vs built-up density'),
]

urban_val = []
for xcol, ycol, label in pairs:
    if xcol not in gdf.columns or ycol not in gdf.columns: continue
    sub = gdf[[xcol,ycol]].dropna()
    if len(sub) < 20: continue
    rs, ps = spearmanr(sub[xcol], sub[ycol])
    urban_val.append({'pair':label, 'x':xcol, 'y':ycol,
                      'spearman_rho':round(rs,4), 'p':round(ps,5), 'n':len(sub)})
    print(f'  ρ={rs:+.3f} (p={ps:.4f})  n={len(sub)}  {label}')

pd.DataFrame(urban_val).to_csv(TABS / 'validation_urban_consistency.csv', index=False)
print('Saved validation_urban_consistency.csv')

## 4. Indicator Independence (Correlation Matrix)

In [ ]:
ind_cols = [c for c in ['n_BD','n_PD','n_BUGR','n_DRT','n_GW','n_PWA',
            'gw_hazard','sub_hazard','climate_stress','exposure','vulnerability']
            if c in gdf.columns]

df_ind = gdf[ind_cols].dropna()
corr_mat = df_ind.corr(method='spearman')
corr_mat.to_csv(TABS / 'validation_correlation_matrix.csv')

high_corr = [(r,c,round(corr_mat.loc[r,c],3))
             for i,r in enumerate(corr_mat.index)
             for j,c in enumerate(corr_mat.columns)
             if i < j and abs(corr_mat.loc[r,c]) > 0.6]

print(f'Indicator pairs |ρ| > 0.6:')
for r,c,v in high_corr: print(f'  {r} × {c}: {v:+.3f}')
if not high_corr: print('  None — indicators are reasonably independent.')

fig, ax = plt.subplots(figsize=(9,7))
im = ax.imshow(corr_mat.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_mat.columns)))
ax.set_yticks(range(len(corr_mat.index)))
ax.set_xticklabels(corr_mat.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(corr_mat.index, fontsize=8)
plt.colorbar(im, ax=ax, label='Spearman ρ')
for i in range(len(corr_mat.index)):
    for j in range(len(corr_mat.columns)):
        v = corr_mat.values[i,j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=6.5,
                color='white' if abs(v)>0.5 else 'black')
ax.set_title('Spearman correlation matrix of CGSRI indicators\n'
             '(|ρ| > 0.6 flagged)', fontsize=10, fontweight='bold')
plt.tight_layout()
fig.savefig(FIGS / 'validation_indicator_correlation.png', dpi=150)
plt.close()
print('Saved validation_indicator_correlation.png')

## 5. CGSRI Sensitivity Analysis

In [ ]:
comp_cols = [c for c in ['gw_hazard','sub_hazard','climate_stress','exposure','vulnerability']
             if c in gdf.columns]
gdf_s = gdf[comp_cols+['cell_id','cgsri','cgsri_class']].dropna(subset=comp_cols).copy()
norm  = gdf_s[comp_cols].apply(lambda s:(s-s.min())/(s.max()-s.min()+1e-12))

n = len(comp_cols)
schemes = {'Equal': np.ones(n)/n}
if n == 5:
    schemes.update({
        'GW-heavy':       np.array([0.40,0.10,0.20,0.15,0.15]),
        'Sub-heavy':      np.array([0.10,0.40,0.20,0.15,0.15]),
        'Exposure-heavy': np.array([0.15,0.15,0.20,0.35,0.15]),
        'Climate-heavy':  np.array([0.15,0.10,0.40,0.20,0.15]),
    })

rank_df  = pd.DataFrame({'cell_id': gdf_s['cell_id'].values})
score_df = pd.DataFrame({'cell_id': gdf_s['cell_id'].values})
for sname, w in schemes.items():
    w = w / w.sum()
    sc = (norm.values * w).sum(axis=1)
    rank_df[sname]  = pd.Series(sc).rank(ascending=False).astype(int).values
    score_df[sname] = sc

print('Rank stability (Spearman ρ vs Equal-weight):')
rho_vals = []
for s in schemes:
    if s == 'Equal': continue
    r, p = spearmanr(rank_df['Equal'], rank_df[s])
    rho_vals.append(r)
    print(f'  vs {s:15s}: ρ={r:.4f}  p={p:.2e}')
mean_stability = float(np.mean(rho_vals)) if rho_vals else float('nan')
print(f'  Mean ρ = {mean_stability:.4f}')

rank_df.to_csv(TABS / 'validation_sensitivity_ranks.csv', index=False)
print('Saved validation_sensitivity_ranks.csv')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
fig.suptitle('CGSRI Sensitivity Analysis — Rank Stability Under Alternative Weightings',
             fontweight='bold', fontsize=11)

# (a) Rank scatter for top-100 cells
ax = axes[0]
colors_s = ['#CC4444','#FF8C00','#2ecc71','#9b59b6']
alt_schemes = [s for s in schemes if s != 'Equal']
for i, s in enumerate(alt_schemes):
    top = rank_df['Equal'] <= 100
    ax.scatter(rank_df.loc[top,'Equal'], rank_df.loc[top,s],
               alpha=0.55, s=14, color=colors_s[i%4], label=s)
ax.plot([1,100],[1,100], 'k--', lw=0.9, label='1:1 line')
ax.set_xlabel('Equal-weight rank'); ax.set_ylabel('Alternative scheme rank')
ax.set_title(f'(a) Rank stability — top 100 cells\n(mean ρ={mean_stability:.3f})', fontsize=9)
ax.legend(fontsize=7)

# (b) Box plots — score distribution for Very High cells
ax = axes[1]
vh_ids = gdf_s.loc[gdf_s['cgsri_class']=='Very high','cell_id'].values
data_vh = [score_df.loc[score_df['cell_id'].isin(vh_ids), s].values
           for s in schemes]
box_c = ['#4477AA','#CC4444','#FF8C00','#2ecc71','#9b59b6']
bp = ax.boxplot(data_vh, labels=list(schemes.keys()), patch_artist=True,
                medianprops=dict(color='black',lw=1.5))
for patch, c in zip(bp['boxes'], box_c):
    patch.set_facecolor(c); patch.set_alpha(0.65)
ax.set_ylabel('CGSRI score')
ax.set_title('(b) Score distribution for Very High cells\nunder each weighting scheme', fontsize=9)
ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
fig.savefig(FIGS / 'validation_sensitivity_analysis.png', dpi=150)
plt.close()
print('Saved validation_sensitivity_analysis.png')

## 6. Spatial Pattern Validation

In [ ]:
spatial_checks = [
    ('elev_m',            'cgsri', 'Elevation (m) vs CGSRI',
     'Expected ρ < 0: lower = higher coastal exposure'),
    ('dist_coast_km',     'cgsri', 'Distance to coast (km) vs CGSRI',
     'Expected ρ < 0: closer coast = higher risk'),
    ('pop_density_km2',   'cgsri', 'Population density vs CGSRI',
     'Expected ρ > 0: more people = higher exposure'),
    ('chirps_mean_rain_mm','cgsri','Mean rainfall (mm/yr) vs CGSRI',
     'Mixed: drier = drought stress; wetter = recharge'),
]

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
fig.suptitle('Spatial Pattern Validation — CGSRI vs independent spatial covariates',
             fontweight='bold', fontsize=11)
axes = axes.flatten()

spatial_val = []
for idx,(xcol,ycol,label,expected) in enumerate(spatial_checks):
    if xcol not in gdf.columns: continue
    sub = gdf[[xcol,ycol]].dropna()
    rs, ps = spearmanr(sub[xcol], sub[ycol])
    rp, pp = pearsonr(sub[xcol], sub[ycol])
    spatial_val.append({'covariate':xcol,'spearman_rho':round(rs,4),'p':round(ps,5),
                        'expected':expected})
    ax = axes[idx]
    ax.scatter(sub[xcol], sub[ycol], c=sub[ycol], cmap='RdYlGn_r',
               s=5, alpha=0.4, linewidths=0)
    m, b, *_ = stats.linregress(sub[xcol], sub[ycol])
    xs = np.array([sub[xcol].min(), sub[xcol].max()])
    ax.plot(xs, m*xs+b, 'k-', lw=1.2)
    ax.set_xlabel(xcol.replace('_',' '), fontsize=8)
    ax.set_ylabel('CGSRI', fontsize=8)
    ax.set_title(f'{label}\nρ={rs:+.3f} (p={ps:.4f})', fontsize=8)
    ax.text(0.97,0.03, expected, transform=ax.transAxes,
            ha='right', va='bottom', fontsize=6, color='navy', style='italic')

plt.tight_layout()
fig.savefig(FIGS / 'validation_spatial_pattern.png', dpi=150)
plt.close()

df_spat = pd.DataFrame(spatial_val)
print(df_spat[['covariate','spearman_rho','p','expected']].to_string(index=False))
df_spat.to_csv(TABS / 'validation_spatial_pattern.csv', index=False)
print('\nSaved validation_spatial_pattern.png + .csv')

## 7. Validation Report

In [ ]:
# ── Build readable validation report ─────────────────────────────────────
rp_gw_pdsi = corr_summary.get('pdsi',  (np.nan,)*4)
rp_gw_rain = corr_summary.get('precip_mm', (np.nan,)*4)

pdsi_ep_file = TABS / 'drought_episodes_pdsi.csv'
ep_lines = []
if pdsi_ep_file.exists():
    for _, row in pd.read_csv(pdsi_ep_file).iterrows():
        ep_lines.append(
            f'  - {row["start"]} \u2013 {row["end"]} ({row["duration_months"]} months, '
            f'min PDSI={row["min_pdsi"]:.2f}, {row["severity"]})')

ep_text          = '\n'.join(ep_lines) if ep_lines else '  No PDSI episodes file found.'
high_corr_text   = '; '.join([f'{r}x{c}={v}' for r,c,v in high_corr]) if high_corr else 'No collinear pairs detected.'
rho_str          = f'{mean_stability:.4f}'
pdsi_r_str       = f'{rp_gw_pdsi[0]:.3f} (p={rp_gw_pdsi[1]:.4f})'
pdsi_rho_str     = f'{rp_gw_pdsi[2]:.3f} (p={rp_gw_pdsi[3]:.4f})'
rain_r_str       = f'{rp_gw_rain[0]:.3f} (p={rp_gw_rain[1]:.4f})'
n_high_corr_str  = str(len(high_corr))

report = (
    '# Validation Report — Ghana Groundwater\u2013Subsidence Pipeline\n\n'
    f'**Generated:** {pd.Timestamp.now().strftime("%Y-%m-%d")}  \n'
    '**Type:** Remote sensing cross-validation (no borehole data available)\n\n'
    '---\n\n'
    '## 1. Temporal Validation\n\n'
    '| Metric | Value |\n'
    '|--------|-------|\n'
    f'| GWSA\u2013PDSI Pearson r (lag=0) | {pdsi_r_str} |\n'
    f'| GWSA\u2013PDSI Spearman \u03c1 | {pdsi_rho_str} |\n'
    f'| GWSA\u2013Rainfall Pearson r | {rain_r_str} |\n\n'
    'Moderate positive GWSA\u2013PDSI correlation is physically plausible.\n'
    'See `outputs/tables/validation_lag_correlations.csv` for lag analysis.\n\n'
    '## 2. Drought Episode Cross-Reference\n\n'
    'PDSI drought episodes (PDSI < \u22121.0 for \u22652 consecutive months):\n'
    f'{ep_text}\n\n'
    'The 2004\u20132010 episode aligns with documented West Africa drought.\n'
    'The 2002\u20132003 episode aligns with the northern Ghana drought emergency.\n\n'
    '## 3. Urban Growth Consistency\n\n'
    'See `outputs/tables/validation_urban_consistency.csv`.\n\n'
    '## 4. Indicator Independence\n\n'
    f'Indicator pairs with |Spearman \u03c1| > 0.6: **{n_high_corr_str}**  \n'
    f'{high_corr_text}\n\n'
    'Full matrix: `outputs/tables/validation_correlation_matrix.csv`\n\n'
    '## 5. Sensitivity Analysis\n\n'
    f'Mean Spearman \u03c1 across alternative schemes vs Equal: **{rho_str}**\n\n'
    '\u03c1 > 0.90 = high stability; 0.75\u20130.90 = moderate; < 0.75 = weight-sensitive.\n\n'
    '## 6. Known Limitations\n\n'
    '1. No borehole water-level records for direct GWSA validation.\n'
    '2. GRACE resolution (~330 km) prevents cell-level groundwater validation.\n'
    '3. proxy_subsidence_potential_grid.nc has all-NaN; sub_hazard uses elevation proxy.\n'
    '4. PDSI spatial mean includes drier northern Ghana.\n'
    '5. Ashanti region: no cells in analysis grid.\n'
    '6. Sentinel-1 InSAR not yet processed (pending ASF HyP3 submission).\n\n'
    '## 7. Recommended Follow-up\n\n'
    '1. Request borehole records from Water Resources Commission.\n'
    '2. Submit Sentinel-1 via scripts/submit_hyp3_accra.py.\n'
    '3. Cross-check Very High CGSRI cells against WRC drought vulnerability reports.\n'
    '4. Key informant interviews in >=3 highest-CGSRI districts.\n'
)

(DOCS / 'validation_report.md').write_text(report)
print('Saved docs/validation_report.md')

# ── Final check ────────────────────────────────────────────────────────
outputs = [
    FIGS/'validation_gwsa_vs_climate.png',
    FIGS/'validation_indicator_correlation.png',
    FIGS/'validation_sensitivity_analysis.png',
    FIGS/'validation_spatial_pattern.png',
    TABS/'validation_lag_correlations.csv',
    TABS/'validation_drought_crossref.csv',
    TABS/'validation_correlation_matrix.csv',
    TABS/'validation_sensitivity_ranks.csv',
    TABS/'validation_urban_consistency.csv',
    TABS/'validation_spatial_pattern.csv',
    DOCS/'validation_report.md',
]
print('\n=== Notebook 11 Output Check ===')
for p in outputs:
    status = 'OK' if p.exists() else 'MISSING'
    sz = f'({p.stat().st_size//1024} KB)' if p.exists() else ''
    print(f'  {status}  {p.relative_to(ROOT)}  {sz}')
